# COCO Captions 2014 — bản tối ưu

Primary CIDEr ↑; chỉ torchvision đọc COCO, dữ liệu ảnh/annotations cần tải riêng. Encoder ImageNet + attention/LSTM scratch.

Chạy độc lập: **Restart Kernel → Run All**. Các trọng số ngôn ngữ/tokenizer đều học từ train. Không đọc notebook khác, fewshot hay workbook. Chưa huấn luyện đầy đủ; không kèm điểm benchmark đã đo.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện

Giữ PyTorch/torchvision tương thích của kernel. Cài pycocotools/pycocoevalcap nếu thiếu; CIDEr dùng PTBTokenizer và cần Java/JRE trong JAVA_EXECUTABLE.


In [ ]:
import importlib.util
import subprocess
import sys
NEEDED_PACKAGES = [('pycocotools', 'pycocotools==2.0.11'), ('pycocoevalcap', 'pycocoevalcap==1.2')]
missing = [package for module, package in NEEDED_PACKAGES if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Cần kernel có PyTorch và torchvision tương thích; cài bản CPU/CUDA phù hợp rồi chạy lại.")

from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import io
import json
import math
import random
import re
import unicodedata
import numpy as np
from PIL import Image, ImageOps
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


## 2. Cấu hình cố định

Thiết kế GPU CUDA32GB với hằng số trực tiếp. Chỉ encoder ảnh của bản tối ưu dùng ImageNet; tokenizer và decoder đều scratch. ImageNet classifier không sinh caption/chart summary.


In [ ]:
SEED = 42
VARIANT = "optimized"
ARTIFACT_DIR = Path("artifacts/caption_coco2014") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "data_cache"
WEIGHTS_CACHE_DIR = ARTIFACT_DIR / "weights"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
PRETRAINING_POLICY = "image_components"
PRETRAINED_COMPONENTS = [{'source': 'torchvision', 'model_id': 'resnet50', 'revision': 'ResNet50_Weights.IMAGENET1K_V2', 'role': 'image_encoder', 'input': 'RGB image', 'output': 'spatial visual features, no text output', 'training': 'finetune'}]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(WEIGHTS_CACHE_DIR))  # Chỉ đặt nơi cache, không dùng torch.hub.load.
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; PyTorch {torch.__version__}; torchvision {torchvision.__version__}")


SOURCE = {'row': 2, 'task': 'Image Captioning', 'dataset': 'MS COCO Captions 2014', 'url': 'https://docs.pytorch.org/vision/master/generated/torchvision.datasets.CocoCaptions.html', 'reader': 'torchvision.datasets.CocoCaptions', 'split_protocol': 'train2014; val2014 chia image-disjoint validation5000/held-out còn lại'}
DATA_ROOT = Path("data/coco2014")
BATCH_SIZE = 32
EVAL_BATCH_SIZE = BATCH_SIZE
IMAGE_SIZE = 256
MAX_TOKENS = 40
VOCAB_SIZE = 10000
MIN_WORD_FREQUENCY = 3
EMBED_DIM = 256
HIDDEN_DIM = 512
MEMORY_DIM = 256
DROPOUT = 0.3
EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 0.0003
BACKBONE_LEARNING_RATE = 0.00001
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 5.0
PRIMARY_METRIC = "cider"
DIRECTION = "max"
VALIDATION_IMAGE_COUNT = 5000
JAVA_EXECUTABLE = "java"


In [ ]:
def fingerprint(records):
    payload = json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

def open_rgb(value):
    if isinstance(value, Image.Image):
        return ImageOps.exif_transpose(value).convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return ImageOps.exif_transpose(Image.open(io.BytesIO(value["bytes"]))).convert("RGB")
        value = value.get("path")
    if value is None:
        raise ValueError("Thiếu ảnh; cần PIL.Image, bytes ảnh hoặc đường dẫn thật.")
    with Image.open(value) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def to_device(batch):
    return {key: value.to(DEVICE) if torch.is_tensor(value) and key not in ("lengths", "widths", "target_lengths") else value
            for key, value in batch.items()}

def make_loader(dataset, train=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train, num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))


## 3. Dữ liệu và split

Tải ảnh/annotation thủ công. Mọi caption cùng ảnh giữ trong một split. train2014 học, val2014 chia5000 validation và phần còn lại held-out có nhãn; không gọi đây là COCO test chính thức.


In [ ]:
from torchvision.datasets import CocoCaptions
def coco_records(image_root, annotation_file, prefix):
    if not image_root.is_dir() or not annotation_file.is_file():
        raise FileNotFoundError(
            f"Thiếu {image_root} hoặc {annotation_file}. CocoCaptions chỉ đọc dữ liệu. "
            "Tải train2014.zip, val2014.zip và annotations_trainval2014.zip từ cocodataset.org/#download, "
            "giải nén vào DATA_ROOT để có train2014/, val2014/, annotations/. Cần pycocotools.")
    dataset = CocoCaptions(root=str(image_root), annFile=str(annotation_file))
    rows = []
    for image_id in sorted(dataset.ids):
        info = dataset.coco.loadImgs(image_id)[0]
        path = image_root / info["file_name"]
        captions = [ann["caption"] for ann in dataset.coco.loadAnns(dataset.coco.getAnnIds(imgIds=image_id))]
        if not path.is_file() or not captions or any(not isinstance(c,str) or not c.strip() for c in captions):
            raise ValueError(f"Ảnh/caption không đầy đủ: {path}; không bỏ mẫu âm thầm.")
        rows.append({"id":f"{prefix}:{image_id}","image_id":int(image_id),"image":str(path),"refs":captions})
    return rows

train_records = coco_records(DATA_ROOT/"train2014",DATA_ROOT/"annotations/captions_train2014.json","train2014")
val_pool = coco_records(DATA_ROOT/"val2014",DATA_ROOT/"annotations/captions_val2014.json","val2014")
if {r["image_id"] for r in train_records} & {r["image_id"] for r in val_pool}:
    raise ValueError("Image ID trùng giữa train2014 và val2014.")
if len(val_pool)<=VALIDATION_IMAGE_COUNT:
    raise ValueError("val2014 không đủ ảnh để tách validation và held-out; kiểm tra dữ liệu hoặc sửa VALIDATION_IMAGE_COUNT thủ công.")
order=list(range(len(val_pool)))
random.Random(SEED).shuffle(order)
val_ids=set(order[:VALIDATION_IMAGE_COUNT])
val_records=[r for i,r in enumerate(val_pool) if i in val_ids]
test_records=[r for i,r in enumerate(val_pool) if i not in val_ids]
SPLIT_FINGERPRINTS={name:fingerprint([(r["id"],r["refs"]) for r in records])
                    for name,records in (("train",train_records),("validation",val_records),("held_out_val2014",test_records))}
print("Số ảnh train/validation/held-out:",len(train_records),len(val_records),len(test_records))
print("Fingerprint:",SPLIT_FINGERPRINTS)
print("Held-out là một phần val2014 có nhãn, không phải COCO test server hoặc Karpathy split.")


## 4. Tokenizer, ảnh và Dataset

Fit vocabulary chỉ caption train. Mỗi caption train là một cặp; eval mỗi ảnh một lần với đầy đủ gold reference. Letterbox giữ toàn ảnh. Target train có BOS/EOS và PAD mask; chỉ rút target train về MAX_TOKENS, gold đánh giá không cắt.


In [ ]:
PAD_ID,UNK_ID,BOS_ID,EOS_ID=0,1,2,3
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]",unicodedata.normalize("NFC",text).lower())
counts=Counter(token for r in train_records for text in r["refs"] for token in tokenize(text))
words=sorted((w for w,c in counts.items() if c>=MIN_WORD_FREQUENCY),key=lambda w:(-counts[w],w))[:VOCAB_SIZE-4]
vocab={"<PAD>":PAD_ID,"<UNK>":UNK_ID,"<BOS>":BOS_ID,"<EOS>":EOS_ID}
vocab.update({word:i+4 for i,word in enumerate(words)})
id_to_word={i:word for word,i in vocab.items()}
TOKENIZER_STATE={"vocab":vocab,"tokenizer":"NFC lowercase, regex words+punctuation",
                 "pad":PAD_ID,"unk":UNK_ID,"bos":BOS_ID,"eos":EOS_ID}
PREPROCESS_CONFIG={"image_size":IMAGE_SIZE,"image_geometry":"letterbox trắng, giữ toàn ảnh",
                    "image_norm":"ImageNet RGB mean/std","max_train_tokens":MAX_TOKENS,
                    "max_decode_tokens":MAX_TOKENS,"decode":"greedy, dừng EOS"}
def restore_tokenizer(state):
    global vocab,id_to_word
    vocab=state["vocab"]
    id_to_word={i:word for word,i in vocab.items()}

def image_tensor(value):
    image=open_rgb(value)
    scale=min(IMAGE_SIZE/image.width,IMAGE_SIZE/image.height)
    resized=image.resize((max(1,round(image.width*scale)),max(1,round(image.height*scale))),Image.Resampling.BILINEAR)
    canvas=Image.new("RGB",(IMAGE_SIZE,IMAGE_SIZE),"white")
    canvas.paste(resized,((IMAGE_SIZE-resized.width)//2,(IMAGE_SIZE-resized.height)//2))
    tensor=TF.to_tensor(canvas)
    tensor=TF.normalize(tensor,[.485,.456,.406],[.229,.224,.225])
    return tensor

class CaptionDataset(Dataset):
    def __init__(self,records,training=False):
        self.records=records
        self.training=training
        # Mọi caption train được dùng; eval mỗi ảnh đúng một lần với đầy đủ references.
        self.index=[(i,j) for i,r in enumerate(records) for j in range(len(r["refs"]))] if training else [(i,0) for i in range(len(records))]
    def __len__(self):
        return len(self.index)
    def __getitem__(self,index):
        i,j=self.index[index]
        record=self.records[i]
        tokens=tokenize(record["refs"][j])[:MAX_TOKENS-1]
        target=[BOS_ID]+[vocab.get(w,UNK_ID) for w in tokens]+[EOS_ID]
        return image_tensor(record["image"]),torch.tensor(target),record

def collate_batch(items):
    width=max(len(target) for _,target,_ in items)
    tokens=torch.full((len(items),width),PAD_ID,dtype=torch.long)
    for i,(_,target,_) in enumerate(items):tokens[i,:len(target)]=target
    return {"images":torch.stack([image for image,_,_ in items]),"tokens":tokens,
            "ids":[r["id"] for _,_,r in items],"records":[r for _,_,r in items]}

train_loader=make_loader(CaptionDataset(train_records,training=True),train=True)
val_loader=make_loader(CaptionDataset(val_records))
test_loader=make_loader(CaptionDataset(test_records))
truncated=sum(len(tokenize(text))>MAX_TOKENS-1 for r in train_records for text in r["refs"])
print("Vocabulary train:",len(vocab),"; caption train bị rút ngắn:",truncated,"; gold đánh giá giữ đầy đủ.")


## 5. Kiến trúc

Encoder ImageNet layer4 fine-tune, BatchNorm giữ eval, attention không gian + LSTMCell scratch.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
class CaptionModel(nn.Module):
    def __init__(self,cfg,initialize_pretrained=True):
        super().__init__()
        try:
            self.backbone=(resnet50(weights=ResNet50_Weights.IMAGENET1K_V2) if initialize_pretrained else resnet50(weights=None))
        except Exception as exc:
            raise RuntimeError(f"Không tải được đúng ImageNet weights. Kiểm tra mạng/WEIGHTS_CACHE_DIR; không tự thay random. {exc}") from exc
        self.backbone.fc=nn.Identity()
        for name,p in self.backbone.named_parameters():p.requires_grad=name.startswith("layer4.")
        self.projection=nn.Linear(2048,cfg["memory_dim"])
        self.embedding=nn.Embedding(cfg["vocab_size"],cfg["embed_dim"],padding_idx=PAD_ID)
        self.memory_attention=nn.Linear(cfg["memory_dim"],cfg["memory_dim"])
        self.hidden_attention=nn.Linear(cfg["hidden_dim"],cfg["memory_dim"])
        self.attention_score=nn.Linear(cfg["memory_dim"],1)
        self.initial_h=nn.Linear(cfg["memory_dim"],cfg["hidden_dim"])
        self.initial_c=nn.Linear(cfg["memory_dim"],cfg["hidden_dim"])
        self.decoder=nn.LSTMCell(cfg["embed_dim"]+cfg["memory_dim"],cfg["hidden_dim"])
        self.dropout=nn.Dropout(cfg["dropout"])
        self.output=nn.Linear(cfg["hidden_dim"]+cfg["memory_dim"],cfg["vocab_size"])
    def train(self,mode=True):
        super().train(mode)
        # Giữ BatchNorm ImageNet ở eval; layer4 vẫn nhận gradient fine-tune.
        self.backbone.eval()
        return self
    def encode(self,images):
        b=self.backbone
        x=b.maxpool(b.relu(b.bn1(b.conv1(images))))
        x=b.layer4(b.layer3(b.layer2(b.layer1(x))))
        return self.projection(x.flatten(2).transpose(1,2)).relu()
    def start_state(self,memory):
        mean=memory.mean(1)
        return self.initial_h(mean).tanh(),self.initial_c(mean).tanh()
    def step(self,token,state,memory):
        hidden,cell=state
        scores=self.attention_score(torch.tanh(self.memory_attention(memory)+self.hidden_attention(hidden)[:,None])).squeeze(-1)
        context=(scores.softmax(-1).unsqueeze(-1)*memory).sum(1)
        hidden,cell=self.decoder(torch.cat([self.embedding(token),context],-1),(hidden,cell))
        logits=self.output(self.dropout(torch.cat([hidden,context],-1)))
        return logits,(hidden,cell)
    def forward(self,images,tokens):
        memory=self.encode(images); state=self.start_state(memory); outputs=[]
        for t in range(tokens.size(1)):
            logits,state=self.step(tokens[:,t],state,memory); outputs.append(logits)
        return torch.stack(outputs,1)
    @torch.no_grad()
    def generate(self,images,max_tokens):
        memory=self.encode(images); state=self.start_state(memory)
        token=torch.full((len(images),),BOS_ID,device=images.device,dtype=torch.long)
        done=torch.zeros(len(images),device=images.device,dtype=torch.bool); output=[]
        for _ in range(max_tokens):
            logits,state=self.step(token,state,memory)
            logits[:,[PAD_ID,BOS_ID]]=-torch.inf
            token=logits.argmax(-1); token=torch.where(done,torch.full_like(token,EOS_ID),token)
            output.append(token); done|=token==EOS_ID
            if bool(done.all()):break
        return torch.stack(output,1)
def build_model(cfg,initialize_pretrained=True):
    return CaptionModel(cfg,initialize_pretrained)


In [ ]:
MODEL_CONFIG={"vocab_size":len(vocab),"embed_dim":EMBED_DIM,"hidden_dim":HIDDEN_DIM,"memory_dim":MEMORY_DIM,"dropout":DROPOUT}


## 6. Loss, sinh văn bản và metric

Teacher forcing, shift target đúng, bỏ PAD; inference tự hồi quy, chặn PAD/BOS, dừng EOS. Primary CIDEr với mọi reference của ảnh, PTB chuẩn COCO; không nhân100 điểm toolkit.


In [ ]:
def decode_tokens(ids):
    result=[]
    for index in ids:
        if int(index)==EOS_ID:break
        if int(index) not in (PAD_ID,BOS_ID):result.append(id_to_word[int(index)])
    return " ".join(result)
def compute_loss(model,batch):
    tokens=batch["tokens"]
    logits=model(batch["images"],tokens[:,:-1])
    return nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),tokens[:,1:].reshape(-1),ignore_index=PAD_ID)


In [ ]:
import shutil
from pycocoevalcap.cider.cider import Cider
import pycocoevalcap.tokenizer.ptbtokenizer as ptb_module

def ptb_tokenize(mapping):
    executable=shutil.which(JAVA_EXECUTABLE)
    if executable is None and Path(JAVA_EXECUTABLE).is_file():executable=JAVA_EXECUTABLE
    if executable is None:
        raise RuntimeError("CIDEr chuẩn COCO cần Java cho PTBTokenizer. Cài JRE/JDK và đặt JAVA_EXECUTABLE "
                           "thành java hoặc đường dẫn java.exe; không đổi tokenizer/metric tự động.")
    keys=[key for key,captions in mapping.items() for _ in captions]
    sentences="\n".join(text.replace("\n"," ") for captions in mapping.values() for text in captions)
    jar=Path(ptb_module.__file__).parent/ptb_module.STANFORD_CORENLP_3_4_1_JAR
    completed=subprocess.run([str(executable),"-cp",str(jar),"edu.stanford.nlp.process.PTBTokenizer",
                              "-preserveLines","-lowerCase"],input=sentences.encode("utf-8"),
                             stdout=subprocess.PIPE,stderr=subprocess.PIPE,check=True)
    lines=completed.stdout.decode("utf-8").splitlines()
    if len(lines)!=len(keys):raise ValueError("PTBTokenizer trả sai số dòng.")
    result=defaultdict(list)
    for key,line in zip(keys,lines):
        result[key].append(" ".join(w for w in line.split() if w not in ptb_module.PUNCTUATIONS))
    return dict(result)

def score_corpus(records,predictions):
    gold=ptb_tokenize({r["id"]:r["refs"] for r in records})
    guessed=ptb_tokenize({r["id"]:[text] for r,text in zip(records,predictions)})
    cider,per_image=Cider(n=4,sigma=6.0).compute_score(gold,guessed)
    return {"cider":float(cider),"image_count":len(records),"tokenizer":"COCO Stanford PTB lowercase",
            "scale":"pycocoevalcap raw, không nhân 100"}


In [ ]:
@torch.no_grad()
def evaluate(model,loader,return_predictions=False):
    model.eval(); records=[]; predictions=[]
    for cpu_batch in loader:
        batch=to_device(cpu_batch)
        output=model.generate(batch["images"],MAX_TOKENS).cpu().tolist()
        records.extend(batch["records"]); predictions.extend(decode_tokens(ids) for ids in output)
    if not records:raise ValueError("Không có ảnh đánh giá.")
    metrics=score_corpus(records,predictions)
    rows=[{"id":r["id"],"prediction":p,"references":r["refs"]} for r,p in zip(records,predictions)]
    return (metrics,rows) if return_predictions else metrics


## 7. Huấn luyện và checkpoint

AdamW, backbone LR thấp hơn head, cosine LR, clipping, checkpoint theo primary validation tăng. Không chọn theo test hoặc tạo chênh lệch cố định.


In [ ]:


def make_optimizer(model):
    visual, remaining = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (visual if name.startswith("backbone.") else remaining).append(parameter)
    groups = [{"params": remaining, "lr": LEARNING_RATE}]
    if visual:
        groups.append({"params": visual, "lr": BACKBONE_LEARNING_RATE})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)




In [ ]:
model = build_model(MODEL_CONFIG, initialize_pretrained=True).to(DEVICE)
print(f"Tham số học: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = math.inf if DIRECTION == "min" else -math.inf
stale_epochs = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, seen = 0.0, 0
    for cpu_batch in tqdm(train_loader, desc="Huấn luyện", leave=False):
        batch = to_device(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        loss = compute_loss(model, batch)
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss không hữu hạn; kiểm tra nhãn, mask và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        count = len(batch["ids"])
        total += loss.item() * count
        seen += count
    if not seen:
        raise ValueError("Tập huấn luyện rỗng.")
    loss = total / seen
    metrics = evaluate(model, val_loader)
    score = float(metrics[PRIMARY_METRIC])
    if not math.isfinite(score):
        raise ValueError("Metric validation không hữu hạn.")
    scheduler.step()
    print(f"Epoch {epoch:02d}: loss={loss:.4f}; validation={metrics}")
    improved = score < best_score if DIRECTION == "min" else score > best_score
    if improved:
        best_score, stale_epochs = score, 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "tokenizer": TOKENIZER_STATE,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo primary metric validation.")
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.")
            break


## 8. Tải lại và đánh giá held-out

Reload model mới từ toàn bộ checkpoint và vocabulary trước held-out. Runtime tạo metrics/predictions JSON riêng từng biến thể.


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["tokenizer"] == TOKENIZER_STATE
restore_tokenizer(checkpoint["tokenizer"])
# Không tải backbone lần nữa: state_dict chứa đầy đủ trọng số visual và head.
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"vocab_size":len(vocab),"embed_dim":EMBED_DIM,"hidden_dim":HIDDEN_DIM,"memory_dim":MEMORY_DIM,"dropout":DROPOUT}
best_model = build_model(inference_config, initialize_pretrained=False).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
validation_metrics = evaluate(best_model, val_loader)
test_metrics, predictions = evaluate(best_model, test_loader, return_predictions=True)
results = {"epoch": checkpoint["epoch"], "source": SOURCE, "split_fingerprints": SPLIT_FINGERPRINTS,
           "validation": validation_metrics, "held_out_test": test_metrics}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã nạp epoch {checkpoint['epoch']}.")
print("Validation:", validation_metrics)
print("Held-out:", test_metrics)
print("Ví dụ dự đoán:", predictions[0] if predictions else "Không có mẫu.")


## 9. Mô tả ảnh mới

Chỉ truyền ảnh, không truyền caption, title hay bảng gold vào model. Không được xem ví dụ sinh ngẫu nhiên như điểm chất lượng.


In [ ]:
@torch.no_grad()
def describe_image(image):
    best_model.eval()
    output=best_model.generate(image_tensor(image).unsqueeze(0).to(DEVICE),MAX_TOKENS)
    return decode_tokens(output[0].tolist())

print("Mô tả ảnh đầu held-out:",describe_image(test_records[0]["image"]))
